# Gemma-E4B -- Climate generalization pilot

Content-domain + chart-type generalization test (Chapter 7 Limitation 2, Chapter 8 item 4,
supervisor item 7). Solar-vs-wind line chart, fabricated data, climate-framed claim -- see
`climate_pilot/generate_climate_stimuli.py` for the stimulus design rationale. Same protocols
as the main study (baseline single-image like/scroll + logprobs, single-image across the 6
`metrics/realistic` engagement scales + logprobs, full 7x7 paired A/B `metrics` grid), pointed
at `climate_pilot/posts/` instead of `benchmarking/`. 25 posts (not 50/100) -- this is a scoped
pilot, not a full replication.

In [1]:
import sys, subprocess

# 1. Uninstall torchaudio
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

# 2. Install PyTorch with CUDA 12.4
subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

# 3. Install latest transformers and accelerate (allowing pip to pull compatible tokenizers naturally)
subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

print("✅ Installation complete — restart the kernel now")



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.16.0.dev0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.23.1 which is incompatible.


✅ Done — restart the kernel now


Restart kernel after running the setup cell above.

In [2]:
!nvidia-smi

Sat Aug 22 17:03:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:CF:00.0 Off |                   On |
| N/A   29C    P0             77W /  700W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [3]:
# --- HF Auth ---
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

# --- Path setup ---
from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

# --- Load Gemma model ---
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-4-E4B-it"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
device = model.device

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [4]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Configuration: climate pilot, NOT the main benchmarking/ pool ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_climate/  -- shared across all 4 climate-pilot models, so they see the identical 25-image sample
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 25

correct_dir = ROOT_DIR / "climate_pilot/posts/correct/PNGs"
incorrect_dir = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic condition only -- the condition that showed the strongest conformity effect
# in the main study (Section 6.2)
correct_base = ROOT_DIR / "climate_pilot/posts/correct/PNGs/metrics/realistic"
incorrect_base = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs/metrics/realistic"


📋 Loading existing selection from /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/selected_images.json
✅ All selected numbers verified in both correct and incorrect folders.
Selected 25 pairs → 50 images total


In [5]:
import torch
free, total = torch.cuda.mem_get_info(0)
print(torch.cuda.get_device_name(0))
print(f"VRAM total   : {total / 1e9:.1f} GB")
print(f"VRAM free    : {free / 1e9:.1f} GB   (device-wide, all processes)")
print(f"this process : {torch.cuda.memory_reserved(0) / 1e9:.1f} GB reserved")

NVIDIA H100 80GB HBM3 MIG 3g.40gb
VRAM total   : 42.3 GB
VRAM free    : 26.0 GB   (device-wide, all processes)
this process : 15.9 GB reserved


In [6]:
from e1_utils.inference_gemma import run_inference_gemma

In [7]:
from e1_utils.e1_optimized import (
    run_e1_baseline_logprobs, run_e1_metrics_logprobs,
    LIKE_CANDIDATES_SINGLE, LIKE_CANDIDATES_YESNO
)

from e1_utils.inference_gemma import run_inference_with_scores_gemma

## Approach 1 -- single image, like/scroll, baseline (0 engagement)

In [8]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json",
              inference_fn=run_inference_gemma)

📋 Resuming — 50 images already processed.
⏭ Skipping: 001_correct
⏭ Skipping: 001_incorrect
⏭ Skipping: 002_correct
⏭ Skipping: 002_incorrect
⏭ Skipping: 003_correct
⏭ Skipping: 003_incorrect
⏭ Skipping: 004_correct
⏭ Skipping: 004_incorrect
⏭ Skipping: 005_correct
⏭ Skipping: 005_incorrect
⏭ Skipping: 006_correct
⏭ Skipping: 006_incorrect
⏭ Skipping: 007_correct
⏭ Skipping: 007_incorrect
⏭ Skipping: 008_correct
⏭ Skipping: 008_incorrect
⏭ Skipping: 009_correct
⏭ Skipping: 009_incorrect
⏭ Skipping: 010_correct
⏭ Skipping: 010_incorrect
⏭ Skipping: 011_correct
⏭ Skipping: 011_incorrect
⏭ Skipping: 012_correct
⏭ Skipping: 012_incorrect
⏭ Skipping: 013_correct
⏭ Skipping: 013_incorrect
⏭ Skipping: 014_correct
⏭ Skipping: 014_incorrect
⏭ Skipping: 015_correct
⏭ Skipping: 015_incorrect
⏭ Skipping: 016_correct
⏭ Skipping: 016_incorrect
⏭ Skipping: 017_correct
⏭ Skipping: 017_incorrect
⏭ Skipping: 018_correct
⏭ Skipping: 018_incorrect
⏭ Skipping: 019_correct
⏭ Skipping: 019_incorrect
⏭ Skippi

In [9]:
run_e1_baseline_logprobs(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_baseline_logprobs.json", score_fn=run_inference_with_scores_gemma)

✅ 001_correct → like {'like': {'logprob': -19.79867935180664, 'prob_forced_choice': 0.9999972735297293}, 'scroll': {'logprob': -32.61117935180664, 'prob_forced_choice': 2.7264702706959303e-06}}
✅ 001_incorrect → like {'like': {'logprob': -24.256864547729492, 'prob_forced_choice': 0.9997546300515665}, 'scroll': {'logprob': -32.56936264038086, 'prob_forced_choice': 0.00024536994843337746}}
✅ 002_correct → like {'like': {'logprob': -22.72085952758789, 'prob_forced_choice': 0.9999531611157506}, 'scroll': {'logprob': -32.68960952758789, 'prob_forced_choice': 4.6838884249524555e-05}}
✅ 002_incorrect → like {'like': {'logprob': -23.31941032409668, 'prob_forced_choice': 0.999872688715582}, 'scroll': {'logprob': -32.28815841674805, 'prob_forced_choice': 0.00012731128441799367}}
✅ 003_correct → like {'like': {'logprob': -21.925552368164062, 'prob_forced_choice': 0.9999865799864952}, 'scroll': {'logprob': -33.14430236816406, 'prob_forced_choice': 1.3420013504783409e-05}}
✅ 003_incorrect → like {'

In [10]:
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")


Single image analysis: e1_results_baseline
=== Summary ===


,metric,value
0,overall_like_rate_%,98.0
1,like_rate_correct_%,100.0
2,like_rate_incorrect_%,96.0


=== Per Image Results ===


,image,variant,prompt,answer
0,001_correct,correct,You are shown a social media post.\nYou can ei...,like
1,001_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
2,002_correct,correct,You are shown a social media post.\nYou can ei...,like
3,002_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
4,003_correct,correct,You are shown a social media post.\nYou can ei...,like
5,003_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
6,004_correct,correct,You are shown a social media post.\nYou can ei...,like
7,004_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
8,005_correct,correct,You are shown a social media post.\nYou can ei...,like
9,005_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/gemma4-e4b/outputs/e1_analysis_baseline.csv


## Approach 1 variant -- single image, like/scroll, across the 6 `metrics/realistic` engagement scales

In [11]:
run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",
              inference_fn=run_inference_gemma)

📋 Resuming — 300 images already processed.
⏭ Skipping: 001_correct_10
⏭ Skipping: 001_incorrect_10
⏭ Skipping: 002_correct_10
⏭ Skipping: 002_incorrect_10
⏭ Skipping: 003_correct_10
⏭ Skipping: 003_incorrect_10
⏭ Skipping: 004_correct_10
⏭ Skipping: 004_incorrect_10
⏭ Skipping: 005_correct_10
⏭ Skipping: 005_incorrect_10
⏭ Skipping: 006_correct_10
⏭ Skipping: 006_incorrect_10
⏭ Skipping: 007_correct_10
⏭ Skipping: 007_incorrect_10
⏭ Skipping: 008_correct_10
⏭ Skipping: 008_incorrect_10
⏭ Skipping: 009_correct_10
⏭ Skipping: 009_incorrect_10
⏭ Skipping: 010_correct_10
⏭ Skipping: 010_incorrect_10
⏭ Skipping: 011_correct_10
⏭ Skipping: 011_incorrect_10
⏭ Skipping: 012_correct_10
⏭ Skipping: 012_incorrect_10
⏭ Skipping: 013_correct_10
⏭ Skipping: 013_incorrect_10
⏭ Skipping: 014_correct_10
⏭ Skipping: 014_incorrect_10
⏭ Skipping: 015_correct_10
⏭ Skipping: 015_incorrect_10
⏭ Skipping: 016_correct_10
⏭ Skipping: 016_incorrect_10
⏭ Skipping: 017_correct_10
⏭ Skipping: 017_incorrect_10
⏭ Ski

In [12]:
run_e1_metrics_logprobs(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_metrics_logprobs.json", score_fn=run_inference_with_scores_gemma)

✅ 001_correct_10 → like {'like': {'logprob': -19.325101852416992, 'prob_forced_choice': 0.999999738348447}, 'scroll': {'logprob': -34.481353759765625, 'prob_forced_choice': 2.6165155293084287e-07}}
✅ 001_incorrect_10 → like {'like': {'logprob': -20.940114974975586, 'prob_forced_choice': 0.9999991148413813}, 'scroll': {'logprob': -34.87761306762695, 'prob_forced_choice': 8.85158618716305e-07}}
✅ 002_correct_10 → like {'like': {'logprob': -21.873735427856445, 'prob_forced_choice': 0.9999979419548158}, 'scroll': {'logprob': -34.96748733520508, 'prob_forced_choice': 2.05804518420805e-06}}
✅ 002_incorrect_10 → like {'like': {'logprob': -21.21400260925293, 'prob_forced_choice': 0.9999988273601632}, 'scroll': {'logprob': -34.87025451660156, 'prob_forced_choice': 1.172639836744735e-06}}
✅ 003_correct_10 → like {'like': {'logprob': -20.526533126831055, 'prob_forced_choice': 0.9999991684735583}, 'scroll': {'logprob': -34.52653503417969, 'prob_forced_choice': 8.315264416531168e-07}}
✅ 003_incorre

In [13]:
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")


Metrics single image analysis: e1_results_metrics
=== Overall Summary ===


,metric,value
0,overall_like_rate_%,99.67
1,like_rate_correct_%,100.00
2,like_rate_incorrect_%,99.33


=== Rate per Scale Value ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:116: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_scale = df.groupby("scale_value").apply(lambda g: pd.Series({


,scale_value,total_images,overall_like_rate_%,like_rate_correct_%,like_rate_incorrect_%
0,10,50.0,100.0,100.0,100.0
1,100,50.0,100.0,100.0,100.0
2,1000,50.0,100.0,100.0,100.0
3,10000,50.0,100.0,100.0,100.0
4,100000,50.0,100.0,100.0,100.0
5,1000000,50.0,98.0,100.0,96.0


=== Per Image Results ===


,image,num,variant,scale_value,prompt,answer
0,001_correct_10,001,correct,10,You are shown a social media post.\nYou can ei...,like
2,002_correct_10,002,correct,10,You are shown a social media post.\nYou can ei...,like
4,003_correct_10,003,correct,10,You are shown a social media post.\nYou can ei...,like
6,004_correct_10,004,correct,10,You are shown a social media post.\nYou can ei...,like
8,005_correct_10,005,correct,10,You are shown a social media post.\nYou can ei...,like
...,...,...,...,...,...,...
291,021_incorrect_1000000,021,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
293,022_incorrect_1000000,022,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
295,023_incorrect_1000000,023,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
297,024_incorrect_1000000,024,incorrect,1000000,You are shown a social media post.\nYou can ei...,like


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/gemma4-e4b/outputs/e1_analysis_metrics.csv


## Approach 2 -- paired A/B forced choice, full 7x7 `metrics/realistic` disparity grid

In [14]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_gemma)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

📋 Resuming — 712 pairs already processed.
⏭ Skipping pair: 001_correct0_vs_incorrect0
⏭ Skipping pair: 002_correct0_vs_incorrect0
⏭ Skipping pair: 003_correct0_vs_incorrect0
⏭ Skipping pair: 004_correct0_vs_incorrect0
⏭ Skipping pair: 005_correct0_vs_incorrect0
⏭ Skipping pair: 006_correct0_vs_incorrect0
⏭ Skipping pair: 007_correct0_vs_incorrect0
⏭ Skipping pair: 008_correct0_vs_incorrect0
⏭ Skipping pair: 009_correct0_vs_incorrect0
⏭ Skipping pair: 010_correct0_vs_incorrect0
⏭ Skipping pair: 011_correct0_vs_incorrect0
⏭ Skipping pair: 012_correct0_vs_incorrect0
⏭ Skipping pair: 013_correct0_vs_incorrect0
⏭ Skipping pair: 014_correct0_vs_incorrect0
⏭ Skipping pair: 015_correct0_vs_incorrect0
⏭ Skipping pair: 016_correct0_vs_incorrect0
⏭ Skipping pair: 017_correct0_vs_incorrect0
⏭ Skipping pair: 018_correct0_vs_incorrect0
⏭ Skipping pair: 019_correct0_vs_incorrect0
⏭ Skipping pair: 020_correct0_vs_incorrect0
⏭ Skipping pair: 021_correct0_vs_incorrect0
⏭ Skipping pair: 022_correct0_vs_i

In [15]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")


Metrics paired A/B analysis: e1_results_metrics_paired
=== Overall Summary ===


,metric,value
0,overall_liked_correct_%,49.96
1,overall_liked_incorrect_%,50.04
2,invalid_answer_%,0.00


=== Liked Correct Rate per Scale Pair ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:163: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_pair = df.groupby("pair").apply(lambda g: pd.Series({


,pair,total_pairs,liked_correct_%,liked_incorrect_%,invalid_%
0,0_vs_0,25.0,68.0,32.0,0.0
1,0_vs_10,25.0,0.0,100.0,0.0
2,0_vs_100,25.0,0.0,100.0,0.0
3,0_vs_1000,25.0,0.0,100.0,0.0
4,0_vs_10000,25.0,0.0,100.0,0.0
5,0_vs_100000,25.0,0.0,100.0,0.0
6,0_vs_1000000,25.0,0.0,100.0,0.0
7,10_vs_0,25.0,100.0,0.0,0.0
8,10_vs_10,25.0,64.0,36.0,0.0
9,10_vs_100,25.0,0.0,100.0,0.0


=== Per Pair Results ===


,image,num,correct_scale,incorrect_scale,post_a_variant,post_b_variant,prompt,answer,liked_variant,pair
0,001_correct0_vs_incorrect0,001,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
1,002_correct0_vs_incorrect0,002,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
2,003_correct0_vs_incorrect0,003,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
3,004_correct0_vs_incorrect0,004,0,0,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,0_vs_0
4,005_correct0_vs_incorrect0,005,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
...,...,...,...,...,...,...,...,...,...,...
1220,021_correct1000000_vs_incorrect1000000,021,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,1000000_vs_1000000
1221,022_correct1000000_vs_incorrect1000000,022,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,1000000_vs_1000000
1222,023_correct1000000_vs_incorrect1000000,023,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,1000000_vs_1000000
1223,024_correct1000000_vs_incorrect1000000,024,1000000,1000000,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,1000000_vs_1000000


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/gemma4-e4b/outputs/e1_analysis_metrics_paired.csv
